In [29]:
import numpy as np
import pandas as pd
df=pd.read_csv("rahul_transactions.csv")


duration=[]

transaction=[]
tran_no=[]
days=[]
net_change=[]
print("="*100)
print("SpendDNA REPORT - Rahul Sharma")
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

df["Amount"] = df["Amount"].apply(
    lambda x: str(x)
        .replace("₹", "")
        .replace("Rs.", "")
        .replace("Rs", "")
        .replace(",", "")
        .replace(" ", "")
        .strip()
)


df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
df["Type"] = df["Type"].astype(str).str.strip().str.lower()
df["Type"] = df["Type"].replace({
    "dr": "debit",
    "cr": "credit"
})
before=len(df)
df = df.drop_duplicates()
print("Transactions:",len(df))


for date in df["Date"]:
    if date not in days:
        days.append(date)
month=len(days)//30
print("Months:",month)
print("="*100)

print("SUMMARY")

credit = 0
debit = 0

for i in range(len(df)):
    if df["Type"].iloc[i] == "credit":
        credit += df["Amount"].iloc[i]
    elif df["Type"].iloc[i] == "debit":
        debit += df["Amount"].iloc[i]

print("Total credit:", credit)
print("Total debit: ", debit)
net_change=credit-debit
print("Net change: ",net_change)

saving_rate=(net_change/credit)*100
print("Savings rate:",round(saving_rate,2),"%" )
print("Transactions:",len(df))
df["vendor_clean"] = df["Description"].apply(clean_vendor)
print("="*100)
print("TOP CATEGORIES (% of debit total)")
print("="*100)

category_map = {
    "Food Delivery": ["Swiggy", "Zomato"],
    "Quick Commerce": ["Blinkit", "Zepto", "BigBasket", "DMart"],
    "E-commerce": ["Amazon", "Flipkart", "Myntra"],
    "Investments": ["Zerodha"]
}

df["Category"] = "Other"

for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        vendor = df["vendor_clean"].iloc[i]
        for category in category_map:

            if vendor in category_map[category]:
                df.loc[df.index[i], "Category"] = category
                break

#restaurant
restaurant_keywords = [
    "RESTAURANT",
    "DOMINO",
    "MCDONALD",
    "KFC",
    "PIZZA",
    "CAFE",
    "BURGER"
]

for i in range(len(df)):

    if df["Type"].iloc[i] == "debit":
        description = str(df["Description"].iloc[i]).upper()

        for keyword in restaurant_keywords:
            if keyword in description:
                df.loc[df.index[i], "Category"] = "Restaurants"
                break


#to calculate category total
category_totals = {}

for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        category = df["Category"].iloc[i]
        amount = df["Amount"].iloc[i]
        if category not in category_totals:
            category_totals[category] = 0
        category_totals[category] += amount


#to ssort categories by spending
sorted_categories = sorted(
    category_totals.items(),
    key=lambda x: x[1],
    reverse=True
)


#top 5 categories
for category, amount in sorted_categories:
    if category != "Other":
        percentage = (amount / debit) * 100
        bars = "#" * int(percentage)
        print(
            f"{category:<20}"
            f"{bars:<25}"
            f"{percentage:>5.1f}%   "
            f"Rs. {amount:,.0f}"
        )
print("="*100)
print("TOP VENDORS")
print("="*100)

vendor_totals = {}
for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        vendor = df["vendor_clean"].iloc[i]
        amount = df["Amount"].iloc[i]
        if vendor not in vendor_totals:
            vendor_totals[vendor] = 0

        vendor_totals[vendor] += amount


#to sort vendors from highest spending to lowest
sorted_vendors = sorted(
    vendor_totals.items(),
    key=lambda x: x[1],
    reverse=True
)


#to isplay top 10 vendors
for vendor, amount in sorted_vendors[:10]:
    percentage = (amount / debit) * 100
    print(
        f"{vendor:<20}"
        f"{percentage:>5.1f}%   "
        f"Rs. {amount:,.0f}"
    )
print("="*100)
print("TIME-OF-DAY PATTERNS")
print("="*100)

df["Time_clean"] = pd.to_datetime(
    df["Time"],
    errors="coerce",
    format="mixed"

)
df["Hour"] = df["Time_clean"].dt.hour
food_hours = {}
for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        if df["Category"].iloc[i] == "Food Delivery":
            hour = df["Hour"].iloc[i]
            if hour not in food_hours:
                food_hours[hour] = 0
            food_hours[hour] += 1

if len(food_hours) > 0:
    peak_food_hour = max(
        food_hours,
        key=food_hours.get
    )

    #percentage of food-delivery orders
    total_food_orders = 0
    for hour in food_hours:
        total_food_orders += food_hours[hour]

    peak_food_orders = food_hours[peak_food_hour]
    peak_percentage = (
        peak_food_orders / total_food_orders
    ) * 100
    start_hour = peak_food_hour
    end_hour = (peak_food_hour + 4) % 24
    print(
        f"Food Delivery peaks: "
        f"{start_hour:02d}:00 - {end_hour:02d}:00 "
        f"({round(peak_percentage, 0):.0f}% of orders)"
    )

cafe_hours = {}
for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        description = str(
            df["Description"].iloc[i]
        ).upper()

        if (
            "CAFE" in description
            or "COFFEE" in description
        ):

            hour = df["Hour"].iloc[i]
            if hour not in cafe_hours:
                cafe_hours[hour] = 0
            cafe_hours[hour] += 1

if len(cafe_hours) > 0:
    peak_cafe_hour = max(
        cafe_hours,
        key=cafe_hours.get
    )

    cafe_start = peak_cafe_hour
    cafe_end = (peak_cafe_hour + 2) % 24
    print(
        f"Cafe peaks:          "
        f"{cafe_start:02d}:00 - {cafe_end:02d}:00 "
        f"(morning runs)"
    )

quick_hours = {}
for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        if df["Category"].iloc[i] == "Quick Commerce":
            hour = df["Hour"].iloc[i]
            if hour not in quick_hours:
                quick_hours[hour] = 0

            quick_hours[hour] += 1

if len(quick_hours) > 0:
    values = list(quick_hours.values())
    highest = max(values)
    lowest = min(values)
    if highest - lowest <= highest * 0.5:
        quick_result = "evenly distributed"
    else:
        quick_result = "concentrated"

    print(
        f"Quick Commerce:     {quick_result}"
    )
    print("="*100)
print("MONTHLY TREND (Food Delivery)")
print("="*100)

monthly_food = {}
for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        if df["Category"].iloc[i] == "Food Delivery":
            month_name = df["Date"].iloc[i].strftime("%b")
            if month_name not in monthly_food:
                monthly_food[month_name] = 0
            monthly_food[month_name] += df["Amount"].iloc[i]

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]

for month in months:

    amount = monthly_food.get(month, 0)
    bars = "#" * int(amount / 2000)

    print(
        f"{month:<5}"
        f"Rs. {amount:>8,.0f}   "
        f"{bars}"
    )
print("="*100)
print("RAHUL'S SPENDING ARCHETYPES")
print("="*100)

#archetype
food_percentage = 0
if debit > 0:
    food_amount = 0
    for i in range(len(df)):
        if df["Type"].iloc[i] == "debit":
            if df["Category"].iloc[i] == "Food Delivery":
                food_amount += df["Amount"].iloc[i]
    food_percentage = (food_amount / debit) * 100
print(
    "-> THE FOODIE"
    f" ({food_percentage:.1f}% on food)"
)

quick_percentage = 0
if debit > 0:
    quick_amount = 0
    for i in range(len(df)):
        if df["Type"].iloc[i] == "debit":
            if df["Category"].iloc[i] == "Quick Commerce":
                quick_amount += df["Amount"].iloc[i]

    quick_percentage = (quick_amount / debit) * 100
print(
    "-> THE QUICK COMMERCE"
    f" ({quick_percentage:.1f}% on Q-com)"
)

ecommerce_percentage = 0
if debit > 0:
    ecommerce_amount = 0
    for i in range(len(df)):
        if df["Type"].iloc[i] == "debit":
            if df["Category"].iloc[i] == "E-commerce":
                ecommerce_amount += df["Amount"].iloc[i]

    ecommerce_percentage = (ecommerce_amount / debit) * 100
print(
    "-> THE SHOPAHOLIC"
    f" ({ecommerce_percentage:.1f}% on e-commerce)"
)

investment_percentage = 0
if debit > 0:
    investment_amount = 0
    for i in range(len(df)):
        if df["Type"].iloc[i] == "debit":
            if df["Category"].iloc[i] == "Investments":
                investment_amount += df["Amount"].iloc[i]

    investment_percentage = (investment_amount / debit) * 100
print(
    "-> THE INVESTOR"
    f" ({investment_percentage:.1f}% on SIPs)"
)

late_food_orders = 0
total_food_orders = 0
for i in range(len(df)):
    if df["Type"].iloc[i] == "debit":
        if df["Category"].iloc[i] == "Food Delivery":
            total_food_orders += 1
            hour = df["Hour"].iloc[i]
            if hour >= 21 or hour < 1:
                late_food_orders += 1
late_food_percentage = 0
if total_food_orders > 0:
    late_food_percentage = (
        late_food_orders / total_food_orders
    ) * 100

print(
    "-> THE LATE-NIGHT SNACKER"
    f" ({late_food_percentage:.0f}% food after 9 PM)"
)
print(
    "-> THE YOLO SPENDER"
    f" (savings rate {saving_rate:.0f}%)"
)
print("="*100)
print("KEY INSIGHTS")
print("="*100)

months = []
for date in df["Date"]:
    current_month = date.to_period("M")
    if current_month not in months:
        months.append(current_month)

number_of_months = len(months)
monthly_change = net_change / number_of_months
print(
    f"1. Rahul is burning through his savings at "
    f"Rs. {abs(monthly_change):,.0f} per month."
)

print(
    f"   - {abs(saving_rate):.0f}% "
    f"{'over his salary' if net_change < 0 else 'saved from his salary'}."
)
print(
    f"2. {late_food_percentage:.0f}% of his food-delivery "
    f"orders happen after 9 PM."
)
discretionary_percentage = (
    food_percentage
    + quick_percentage
    + ecommerce_percentage
)
print(
    f"3. Investments ({investment_percentage:.1f}%) "
    f"are a significant part of spending, but discretionary "
    f"categories (Food + Q-com + E-com = "
    f"{discretionary_percentage:.1f}%) "
    f"also represent a large share of his wallet."
)
print("="*100)

SpendDNA REPORT - Rahul Sharma
Transactions: 1310
Months: 6
SUMMARY
Total credit: 509774.0
Total debit:  1678901.0
Net change:  -1169127.0
Savings rate: -229.34 %
Transactions: 1310
TOP CATEGORIES (% of debit total)
E-commerce          ################################# 33.9%   Rs. 568,804
Food Delivery       ########                   8.4%   Rs. 140,538
Quick Commerce      #####                      5.0%   Rs. 84,659
Restaurants         ####                       4.2%   Rs. 70,614
TOP VENDORS
Other                35.3%   Rs. 592,885
Amazon               19.6%   Rs. 328,530
Flipkart             10.2%   Rs. 170,745
Axis Bank             6.3%   Rs. 105,201
Swiggy                5.7%   Rs. 95,523
Myntra                4.1%   Rs. 69,529
Zomato                3.3%   Rs. 55,316
HDFC                  3.3%   Rs. 54,569
Cash Withdrawal       2.7%   Rs. 45,500
Zepto                 1.7%   Rs. 28,157
TIME-OF-DAY PATTERNS
Food Delivery peaks: 20:00 - 00:00 (12% of orders)
Cafe peaks:          10:00